# Superstore Sales Analysis

**Источник:** Kaggle, Superstore Sales Dataset  
**Файл:** data/raw/train.csv  
**Размер:** 9800 строк, 18 столбцов  
**Пропуски:** 11 в Postal Code (0.1%)

## Столбцы

- Row ID - идентификатор строки
- Order ID - идентификатор заказа
- Order Date - дата заказа
- Ship Date - дата отправки
- Ship Mode - способ доставки
- Customer ID - идентификатор клиента
- Customer Name - имя клиента
- Segment - сегмент клиента (Consumer, Corporate, Home Office)
- Country - страна (только United States)
- City - город
- State - штат
- Postal Code - почтовый индекс (11 пропусков)
- Region - регион (South, West, East, Central)
- Product ID - идентификатор товара
- Category - категория (Furniture, Office Supplies, Technology)
- Sub-Category - подкатегория
- Product Name - название товара
- Sales - сумма продажи, доллары

## Первичные наблюдения

- Sales имеет сильную правостороннюю асимметрию: среднее 230.77, медиана 54.49. Есть крупные заказы.
- Row ID не несёт аналитической ценности.
- Country содержит только одно значение.
- Order Date и Ship Date хранятся как строки, нужна конвертация в datetime.

## Решение по пропускам

11 пропусков в Postal Code (0.1%). Удалены строки, так как потеря незначительна, а Postal Code не участвует в основных гипотезах.

In [13]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

In [14]:
file = '../data/raw/superstore.csv'
data = pd.read_csv(file)
print('Первые 5 строк датасета')
print(data.head(5))

Первые 5 строк датасета
   Row ID        Order ID  Order Date   Ship Date       Ship Mode Customer ID  \
0       1  CA-2017-152156  08/11/2017  11/11/2017    Second Class    CG-12520   
1       2  CA-2017-152156  08/11/2017  11/11/2017    Second Class    CG-12520   
2       3  CA-2017-138688  12/06/2017  16/06/2017    Second Class    DV-13045   
3       4  US-2016-108966  11/10/2016  18/10/2016  Standard Class    SO-20335   
4       5  US-2016-108966  11/10/2016  18/10/2016  Standard Class    SO-20335   

     Customer Name    Segment        Country             City       State  \
0      Claire Gute   Consumer  United States        Henderson    Kentucky   
1      Claire Gute   Consumer  United States        Henderson    Kentucky   
2  Darrin Van Huff  Corporate  United States      Los Angeles  California   
3   Sean O'Donnell   Consumer  United States  Fort Lauderdale     Florida   
4   Sean O'Donnell   Consumer  United States  Fort Lauderdale     Florida   

   Postal Code Region     

In [15]:
print('Информация о датасете')
print(data.info())

Информация о датасете
<class 'pandas.DataFrame'>
RangeIndex: 9800 entries, 0 to 9799
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Row ID         9800 non-null   int64  
 1   Order ID       9800 non-null   str    
 2   Order Date     9800 non-null   str    
 3   Ship Date      9800 non-null   str    
 4   Ship Mode      9800 non-null   str    
 5   Customer ID    9800 non-null   str    
 6   Customer Name  9800 non-null   str    
 7   Segment        9800 non-null   str    
 8   Country        9800 non-null   str    
 9   City           9800 non-null   str    
 10  State          9800 non-null   str    
 11  Postal Code    9789 non-null   float64
 12  Region         9800 non-null   str    
 13  Product ID     9800 non-null   str    
 14  Category       9800 non-null   str    
 15  Sub-Category   9800 non-null   str    
 16  Product Name   9800 non-null   str    
 17  Sales          9800 non-null   float64
dt

In [16]:
print('Основные числовые характеристики датасета\n')
print(data.describe())

Основные числовые характеристики датасета

            Row ID   Postal Code         Sales
count  9800.000000   9789.000000   9800.000000
mean   4900.500000  55273.322403    230.769059
std    2829.160653  32041.223413    626.651875
min       1.000000   1040.000000      0.444000
25%    2450.750000  23223.000000     17.248000
50%    4900.500000  58103.000000     54.490000
75%    7350.250000  90008.000000    210.605000
max    9800.000000  99301.000000  22638.480000


In [17]:
print(data.isna().sum())
print(f'Количество пропусков всего: {data.isna().sum().sum()}')

Row ID            0
Order ID          0
Order Date        0
Ship Date         0
Ship Mode         0
Customer ID       0
Customer Name     0
Segment           0
Country           0
City              0
State             0
Postal Code      11
Region            0
Product ID        0
Category          0
Sub-Category      0
Product Name      0
Sales             0
dtype: int64
Количество пропусков всего: 11


In [29]:
os.makedirs('../data/processed', exist_ok=True)

data_clean = data.dropna(subset=['Postal Code'])

print(f"Было строк: {len(data)}")
print(f"Стало строк: {len(data_clean)}")
print(f"Удалено: {len(data) - len(data_clean)}")

data_clean.to_csv('../data/processed/superstore_clean.csv', index=False)

print("Сохранено в data/processed/superstore_clean.csv")
print(f"Строк: {len(data_clean)}, Колонок: {len(data_clean.columns)}")

Было строк: 9800
Стало строк: 9789
Удалено: 11
Сохранено в data/processed/superstore_clean.csv
Строк: 9789, Колонок: 18


### Очистка Postal Code
Обнаружено 11 пропусков (0.1% от всех данных).
Решение: удалить строки с пропусками, так как:
1. Потеря незначительна (0.1%).
2. Postal Code не участвует в основных гипотезах.
3. Заполнение модой исказило бы геоданные.

In [30]:
data_clean['Order Date'] = pd.to_datetime(data_clean['Order Date'], format='%d/%m/%Y', errors='coerce')
data_clean['Ship Date'] = pd.to_datetime(data_clean['Ship Date'], format='%d/%m/%Y', errors='coerce')

print(data_clean['Order Date'].head(5))
print(data_clean['Order Date'].dtype)
print("NaT в Order Date:", data_clean['Order Date'].isnull().sum())
print("NaT в Ship Date:", data_clean['Ship Date'].isnull().sum())

0   2017-11-08
1   2017-11-08
2   2017-06-12
3   2016-10-11
4   2016-10-11
Name: Order Date, dtype: datetime64[us]
datetime64[us]
NaT в Order Date: 0
NaT в Ship Date: 0


In [ ]:
print("Тип Postal Code:", data_clean['Postal Code'].dtype)
print("Примеры:", data_clean['Postal Code'].head(10).tolist())
print("Пропуски:", data_clean['Postal Code'].isnull().sum())

Тип Postal Code: float64
Примеры: [42420.0, 42420.0, 90036.0, 33311.0, 33311.0, 90032.0, 90032.0, 90032.0, 90032.0, 90032.0]
Пропуски: 0


In [34]:
print('Sales: min:', data_clean['Sales'].min())
print('Sales: max:', data_clean['Sales'].max())

Sales: min: 0.444
Sales: max: 22638.48


In [35]:
data_clean['Postal Code'] = data_clean['Postal Code'].astype(int)
print(data_clean['Postal Code'].head(5))
print(data_clean['Postal Code'].dtype)

0    42420
1    42420
2    90036
3    33311
4    33311
Name: Postal Code, dtype: int64
int64


In [39]:
data_clean['Order Year'] = data_clean['Order Date'].dt.year
data_clean['Order Month'] = data_clean['Order Date'].dt.month
data_clean['Order Month Name'] = data_clean['Order Date'].dt.month_name()
data_clean['Order Day of Week'] = data_clean['Order Date'].dt.day_name()
data_clean['Order Quarter'] = data_clean['Order Date'].dt.quarter

data_clean['Shipping Days'] = (data_clean['Ship Date'] - data_clean['Order Date']).dt.days
print(data_clean[['Order Date', 'Order Year', 'Order Month', 'Order Month Name', 'Order Day of Week', 'Order Quarter', 'Shipping Days']].head())

  Order Date  Order Year  Order Month Order Month Name Order Day of Week  \
0 2017-11-08        2017           11         November         Wednesday   
1 2017-11-08        2017           11         November         Wednesday   
2 2017-06-12        2017            6             June            Monday   
3 2016-10-11        2016           10          October           Tuesday   
4 2016-10-11        2016           10          October           Tuesday   

   Order Quarter  Shipping Days  
0              4              3  
1              4              3  
2              2              4  
3              4              7  
4              4              7  


Отрицательных Shipping Days: 0


In [42]:
data_clean.to_csv('../data/processed/superstore_clean.csv', index = False)
print(f'Строк: {len(data_clean)}')
print(f'Колонок: {len(data_clean.columns)}')

Строк: 9789
Колонок: 24
